# JupyterLite で学ぶ Python 文法 入門チュートリアル

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** 上で、
プログラミング言語 **Python** の基本的な文法を一から学ぶためのチュートリアルです。

## 対象者
- プログラミングをこれから始める方
- Python の文法を基礎から学び直したい方
- NumPy や pandas を学ぶ前に、Python そのものに慣れておきたい方

## このチュートリアルで学ぶこと
0. JupyterLite の準備（ライブラリの読み込み方・日本語フォントの設定）
1. はじめての Python（print と計算）
2. 変数とデータ型
3. 文字列
4. リスト
5. タプル・辞書・集合
6. 条件分岐（if 文）
7. 繰り返し（for 文・while 文）
8. 関数
9. エラーと例外処理
10. モジュールと標準ライブラリ
11. ファイルの読み書き
12. クラス入門
13. まとめと総合演習

## 使い方
- セルを上から順に `Shift + Enter` で実行してください。途中を飛ばすと、変数が未定義でエラーになることがあります。
- 各章の最後に **練習問題** があります。「解答欄」のセルに自分でコードを書いてから、「解答例」を開いて確認しましょう。
- コードを自由に書き換えて、結果がどう変わるか試してみるのが上達の近道です。

---
## 0. JupyterLite の準備

### 0.1 JupyterLite と通常の Jupyter の違い

JupyterLite は、Python の実行環境（**Pyodide**：WebAssembly 版の Python）を **ブラウザの中で** 動かします。
サーバーもインストールも不要ですが、通常の Jupyter（JupyterLab / Jupyter Notebook）とはいくつか違いがあります。

| 項目 | 通常の Jupyter | JupyterLite |
|---|---|---|
| Python の実行場所 | PC 上の Python | ブラウザ内の Pyodide |
| ライブラリの追加 | ターミナルで `pip install` | ノートブック内で `piplite.install()` や `%pip install` |
| 使えるライブラリ | ほぼすべて | Pyodide 対応のもの（NumPy、pandas、matplotlib など主要なものは使える） |
| ファイルの保存先 | PC のディスク | ブラウザのローカルストレージ |
| インストールの永続性 | 残る | **カーネルを再起動すると消える**（毎回インストールのセルを実行する） |

### 0.2 ライブラリの読み込み方

JupyterLite でライブラリを使うときは、ライブラリを次の 3 種類に分けて考えると迷いません。

**(1) 標準ライブラリ（`math`、`random`、`datetime`、`csv` など）**
Python に最初から付属しているので、`import` するだけで使えます。

```python
import math
print(math.pi)
```

**(2) Pyodide に同梱されている主要ライブラリ（NumPy、pandas、matplotlib、SciPy、scikit-learn など）**
`piplite.install()` で読み込んでから `import` します。ブラウザにライブラリ本体（wheel ファイル）をダウンロードするため、
初回は数秒〜数十秒かかります。

```python
import piplite
await piplite.install(["numpy", "pandas", "matplotlib"])

import numpy as np
```

`await` を付け忘れないでください。「ダウンロードが終わるのを待ってから次に進む」という意味です。

**(3) PyPI で配布されている純 Python 製のライブラリ（`japanize-matplotlib-jlite` など）**
これも `piplite.install()` で PyPI から取得できます。ただし、C 言語などで書かれた拡張モジュールを含むライブラリは、
Pyodide 側でビルド済みのものしか使えません。

#### 書き方のバリエーション

| 書き方 | 説明 |
|---|---|
| `await piplite.install("numpy")` | 1 つだけインストール |
| `await piplite.install(["numpy", "pandas"])` | まとめてインストール（リストで指定） |
| `%pip install numpy` | pip 風のマジックコマンド。中身は `piplite` と同じ |
| `await micropip.install("numpy")` | Pyodide 本体の仕組み。`piplite` はこれを内部で使っている |

#### ローカルの Jupyter でも動くようにする書き方

このサイトのノートブックでは、次のように `try / except` で囲む書き方を採用しています。
`piplite` は JupyterLite にしか存在しないので、PC 上の Jupyter で開いたときは `ImportError` になり、
`pass`（何もしない）で先に進みます。

```python
try:
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib"])
except ImportError:
    pass  # ローカルの Jupyter ではここを通る
```

#### 注意点
- **カーネルを再起動するとインストール結果は消えます。** ページを開き直したときは、必ずインストールのセルから実行し直してください。
- `piplite.install()` にはインターネット接続が必要です（wheel ファイルを CDN や PyPI から取得します）。
- 対応していないライブラリを指定すると `ValueError: Can't find a pure Python 3 wheel for ...` のようなエラーになります。その場合、そのライブラリは JupyterLite では使えません。
- インストール済みのライブラリをもう一度 `piplite.install()` しても害はありません（すぐに終わります）。

それでは実際にインストールしてみましょう。このノートブックでは、第 0 章と最後の総合演習でグラフを描くために
`numpy`、`matplotlib`、日本語フォント用の `japanize-matplotlib-jlite` を使います。

In [ ]:
# JupyterLite 用のパッケージインストール（初回は数十秒かかることがあります）
try:
    import piplite
    await piplite.install(["numpy", "matplotlib", "japanize-matplotlib-jlite"])
    print("piplite でのインストールが完了しました")
except ImportError:
    print("piplite がない環境（ローカルの Jupyter）なのでスキップしました")

インストールできたか確認します。`sys.platform` が `"emscripten"` なら、ブラウザ内の Pyodide で動いています。

In [ ]:
import sys
import numpy as np
import matplotlib

print("Python バージョン   :", sys.version.split()[0])
print("NumPy バージョン    :", np.__version__)
print("matplotlib バージョン:", matplotlib.__version__)
print("実行環境           :", "JupyterLite (Pyodide)" if sys.platform == "emscripten" else "通常の Python")

### 0.3 日本語フォントの設定

matplotlib でグラフを描くとき、標準のフォント（DejaVu Sans）には日本語の文字が含まれていません。
そのため、タイトルや軸ラベルに日本語を使うと、文字が **□□□（いわゆる「豆腐」）** になってしまいます。

JupyterLite（ブラウザ内の Python）は PC にインストールされている日本語フォントを参照できないため、
**日本語フォントを同梱したライブラリを読み込む** のが最も簡単な解決策です。

#### 方法 1（推奨）: `japanize-matplotlib-jlite` を使う

`japanize-matplotlib-jlite` は、日本語フォント（IPAex ゴシック）を同梱した純 Python パッケージです。
`import` するだけで matplotlib の既定フォントが日本語対応のものに切り替わります。

手順:
1. `piplite.install("japanize-matplotlib-jlite")` でインストールする（0.2 のセルで実行済み）
2. `import matplotlib.pyplot as plt` の **後に** `import japanize_matplotlib_jlite` と書く

パッケージ名はハイフン区切り（`japanize-matplotlib-jlite`）、`import` するモジュール名はアンダースコア区切り
（`japanize_matplotlib_jlite`）と、名前が少し違う点に注意してください。

In [ ]:
import matplotlib.pyplot as plt
import japanize_matplotlib_jlite  # 日本語フォントを有効にする（plt の後に import する）

# 日本語を含むグラフを描いてみる
months = ["1月", "2月", "3月", "4月", "5月", "6月"]
sales = [120, 135, 150, 128, 170, 190]

plt.figure(figsize=(7, 4))
plt.plot(months, sales, marker="o")
plt.title("月別売上の推移")
plt.xlabel("月")
plt.ylabel("売上（万円）")
plt.grid(True)
plt.show()

タイトルや軸ラベルの日本語が正しく表示されていれば成功です。
現在 matplotlib がどのフォントを使っているかは、次のように確認できます。

In [ ]:
from matplotlib import font_manager

print("現在のフォント設定:", plt.rcParams["font.family"])

# matplotlib が認識しているフォントのうち、日本語対応（IPA）のものを表示
japanese_fonts = sorted({f.name for f in font_manager.fontManager.ttflist if "IPA" in f.name})
print("利用できる日本語フォント:", japanese_fonts)

#### 方法 2: フォントファイルを自分で用意する

好きなフォント（例: Noto Sans JP）を使いたい場合は、フォントファイル（`.ttf` / `.otf`）を JupyterLite の
ファイルブラウザにアップロードしてから、次のように登録します（ここでは説明のみ）。

```python
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "NotoSansJP-Regular.ttf"                    # アップロードしたフォントファイル
font_manager.fontManager.addfont(font_path)             # matplotlib に登録
font_name = font_manager.FontProperties(fname=font_path).get_name()  # フォント名を調べる
plt.rcParams["font.family"] = font_name                 # 既定フォントに設定
```

`plt.rcParams["font.family"]` に指定するのは **ファイル名ではなくフォント名** です。

#### よくあるトラブル

| 症状 | 原因と対処 |
|---|---|
| 日本語が □ になる | `import japanize_matplotlib_jlite` を実行していない。カーネル再起動後は再実行が必要 |
| `ModuleNotFoundError: No module named 'japanize_matplotlib_jlite'` | `piplite.install("japanize-matplotlib-jlite")` が実行されていない |
| マイナス記号が □ になる | `plt.rcParams["axes.unicode_minus"] = False` を追加する |
| seaborn でフォントが戻ってしまう | `sns.set_theme()` などの **後に** `import japanize_matplotlib_jlite` を実行する |

#### 補足: JupyterLab の画面自体を日本語にする

グラフの日本語表示とは別に、メニューやボタンの表示を日本語にすることもできます。
このサイトには日本語 UI パック（`jupyterlab-language-pack-ja-JP`）が組み込まれているので、
メニューの **Settings → Language → 日本語 (Japanese)** を選んでください。

### 0.4 この先の章について

第 1 章から第 12 章までは Python の基本文法だけを扱うので、追加のライブラリは必要ありません。
最後の総合演習でグラフを描くときに、上で設定した日本語フォントを使います。

---
## 1. はじめての Python

### 1.1 print で文字を表示する

`print()` は、かっこの中の値を画面に表示する **関数** です。文字列（文字の並び）は `"`（ダブルクォート）
または `'`（シングルクォート）で囲みます。

In [ ]:
print("こんにちは、Python!")

In [ ]:
# 複数の値をカンマで区切ると、スペースで区切って表示される
print("Python", "は", "楽しい")

# sep で区切り文字、end で末尾の文字を変えられる
print("2026", "08", "30", sep="-")
print("改行しない", end=" → ")
print("次の出力")

### 1.2 コメント

`#` から行末までは **コメント** として無視されます。コードの説明やメモを書くために使います。

In [ ]:
# これはコメントです。実行されません
print("コメントは無視されます")  # 行の途中からでも書けます

"""
複数行の説明を書きたいときは、三重引用符で囲む方法もあります。
（正確には「文字列」ですが、コメント代わりによく使われます）
"""
print("三重引用符の中も実行には影響しません")

### 1.3 Python を電卓として使う

Python は数値計算がそのまま書けます。

| 演算子 | 意味 | 例 | 結果 |
|:---:|---|---|---|
| `+` | 足し算 | `7 + 3` | `10` |
| `-` | 引き算 | `7 - 3` | `4` |
| `*` | 掛け算 | `7 * 3` | `21` |
| `/` | 割り算（結果は常に小数） | `7 / 2` | `3.5` |
| `//` | 割り算の商（切り捨て） | `7 // 2` | `3` |
| `%` | 割り算の余り | `7 % 2` | `1` |
| `**` | べき乗 | `2 ** 10` | `1024` |

In [ ]:
print(7 + 3)    # 足し算
print(7 - 3)    # 引き算
print(7 * 3)    # 掛け算
print(7 / 2)    # 割り算（結果は常に小数）
print(7 // 2)   # 商（切り捨て）
print(7 % 2)    # 余り
print(2 ** 10)  # べき乗

In [ ]:
# 計算の優先順位は数学と同じ。かっこで変更できる
print(2 + 3 * 4)
print((2 + 3) * 4)

Jupyter では、セルの **最後の行が式** なら `print()` を書かなくても結果が表示されます。

In [ ]:
100 * 1.1

### 練習問題 1

1. `print()` を使って「太郎さん、こんにちは」のように、自分の名前を含む挨拶を表示してください。
2. 1 週間は何秒か計算して表示してください（1 日 = 24 時間、1 時間 = 60 分、1 分 = 60 秒）。
3. 1234 を 7 で割ったときの商と余りを表示してください。

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```python
# 1
print("太郎さん、こんにちは")

# 2
print(7 * 24 * 60 * 60)

# 3
print(1234 // 7, 1234 % 7)
```

</details>

---
## 2. 変数とデータ型

### 2.1 変数

**変数** は値に名前を付けて保存しておく箱のようなものです。`=` で値を代入します
（数学の「等しい」ではなく「右の値を左の名前に入れる」という意味です）。

In [ ]:
price = 120          # 変数 price に 120 を代入
quantity = 3
total = price * quantity
print(total)

In [ ]:
# 変数は上書きできる
count = 1
print(count)
count = count + 1    # 今の値に 1 を足して、count に入れ直す
print(count)
count += 1           # count = count + 1 の省略形（-=, *=, /= もある）
print(count)

#### 変数名のルール
- 英字・数字・アンダースコア `_` が使える（先頭に数字は不可）
- 大文字と小文字は区別される（`Total` と `total` は別の変数）
- `if`、`for`、`print` などの予約語・組み込み関数名は避ける
- 単語をアンダースコアでつなぐ **スネークケース**（`total_price`）が Python の慣習
- `x` や `a` より、`price` や `student_count` のように **意味のわかる名前** を付ける

In [ ]:
# 複数の変数に同時に代入
x, y = 10, 20
print(x, y)

# 値の入れ替え（一時変数が不要）
x, y = y, x
print(x, y)

### 2.2 データ型

値には **型**（データの種類）があります。`type()` で確認できます。

| 型 | 説明 | 例 |
|---|---|---|
| `int` | 整数 | `20`, `-3`, `0` |
| `float` | 浮動小数点数（小数） | `170.5`, `3.0`, `1e-3` |
| `str` | 文字列 | `"山田"`, `'abc'` |
| `bool` | 真偽値 | `True`, `False` |

In [ ]:
age = 20             # int（整数）
height = 170.5       # float（小数）
name = "山田"         # str（文字列）
is_student = True    # bool（真偽値）

print(type(age))
print(type(height))
print(type(name))
print(type(is_student))

In [ ]:
# 型が違うと演算の意味が変わる
print(1 + 2)        # 数値の足し算
print("1" + "2")    # 文字列の連結
# print(1 + "2")    # ← エラー（TypeError）。数値と文字列は足せない

### 2.3 型変換

`int()`、`float()`、`str()` などで型を変換できます。

In [ ]:
num_str = "100"
num = int(num_str)         # 文字列 → 整数
print(num + 50)

print(float("3.14") * 2)   # 文字列 → 小数
print(str(42) + "個")       # 数値 → 文字列
print(int(3.99))           # 小数 → 整数（切り捨て）
print(round(3.14159, 2))   # 四捨五入（小数第 2 位まで）

In [ ]:
# bool への変換：0 や空の文字列は False、それ以外は True
print(bool(0), bool(1), bool(""), bool("abc"))

### 練習問題 2

1. 半径 5 の円の面積を、変数 `radius` と `area` を使って計算し表示してください（円周率は 3.14 とします）。
2. 文字列 `"25"` と `"17"` をそれぞれ整数に変換してから足し算し、結果を表示してください。
3. 変数 `a = 3`、`b = 7` の値を入れ替えて、`a` と `b` を表示してください。

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```python
# 1
radius = 5
area = 3.14 * radius ** 2
print(area)

# 2
print(int("25") + int("17"))

# 3
a, b = 3, 7
a, b = b, a
print(a, b)
```

</details>

---
## 3. 文字列

### 3.1 文字列の作成

文字列は `"` か `'` で囲みます。三重引用符 `"""` を使うと複数行の文字列が書けます。

In [ ]:
s1 = "ダブルクォート"
s2 = 'シングルクォート'
s3 = """複数行の
文字列も
書けます"""
print(s1)
print(s2)
print(s3)

改行やタブなど、そのままでは書けない文字は **エスケープシーケンス** で表します。

| 記号 | 意味 |
|---|---|
| `\n` | 改行 |
| `\t` | タブ |
| `\"` `\'` | 引用符そのもの |
| `\\` | バックスラッシュそのもの |

In [ ]:
print("1行目\n2行目")
print("名前\t点数")
print("彼は \"こんにちは\" と言った")
print('It\'s a pen')

### 3.2 文字列の演算

`+` で連結、`*` で繰り返し、`len()` で文字数を求められます。

In [ ]:
first = "Python"
second = "入門"
print(first + second)        # 連結
print(first + " " + second)
print("-" * 20)              # 繰り返し
print(len(first))            # 文字数
print(len("日本語も1文字ずつ数える"))

### 3.3 インデックスとスライス

文字列の各文字には **0 から始まる番号（インデックス）** が付いています。
負の数を使うと末尾から数えられます（`-1` が最後の文字）。

```
 文字:   P   y   t   h   o   n
 番号:   0   1   2   3   4   5
 負:    -6  -5  -4  -3  -2  -1
```

`[開始:終了]` の形で一部を取り出すことを **スライス** といいます。終了の位置は **含まれない** ことに注意してください。

In [ ]:
word = "Python"
print(word[0])      # 最初の文字
print(word[-1])     # 最後の文字
print(word[0:3])    # 0 番目から 2 番目まで（3 は含まない）
print(word[:2])     # 先頭から 2 文字
print(word[2:])     # 2 番目から末尾まで
print(word[::2])    # 1 つおき
print(word[::-1])   # 逆順

### 3.4 文字列のメソッド

文字列には便利な **メソッド**（値に付属する関数）がたくさんあります。`変数.メソッド名()` の形で呼び出します。

In [ ]:
text = "  Hello, Python World  "
print(text.strip())                # 前後の空白を取り除く
print(text.lower())                # 小文字に
print(text.upper())                # 大文字に
print(text.replace("Python", "Jupyter"))   # 置き換え
print(text.strip().split(" "))     # 空白で分割してリストに
print("Python" in text)            # 含まれているか
print(text.count("o"))             # 出現回数
print(text.find("Python"))         # 位置（見つからなければ -1）

In [ ]:
# 文字列そのものは変更できない（イミュータブル）。メソッドは新しい文字列を返す
name = "python"
name.upper()
print(name)           # 元のまま
name = name.upper()   # 結果を代入し直す
print(name)

In [ ]:
# 判定メソッド（True / False を返す）
print("123".isdigit())
print("abc".isalpha())
print("Hello".startswith("He"))
print("data.csv".endswith(".csv"))

In [ ]:
# join: リストの要素を区切り文字でつないで 1 つの文字列にする
cities = ["東京", "名古屋", "大阪"]
print(", ".join(cities))
print(" → ".join(cities))

### 3.5 f 文字列（フォーマット）

文字列の前に `f` を付け、`{}` の中に変数や式を書くと、その値が埋め込まれます。
`{値:書式}` の形で表示形式も指定できます。

In [ ]:
name = "佐藤"
age = 21
print(f"{name}さんは{age}歳です")
print(f"来年は{age + 1}歳になります")   # {} の中で計算もできる

In [ ]:
pi = 3.14159265
print(f"円周率: {pi:.2f}")          # 小数点以下 2 桁
print(f"{1234567:,}")               # 3 桁区切り
print(f"{0.256:.1%}")               # パーセント表示
print(f"[{'apple':<8}][{'kiwi':^8}][{'melon':>8}]")   # 幅 8 で左寄せ・中央・右寄せ
print(f"{42:05d}")                  # 0 埋めで 5 桁
print(f"{2 ** 10 = }")              # 式と結果を同時に表示（デバッグに便利）

### 練習問題 3

1. `s = "Hello, World"` から、スライスで `"World"` を取り出して表示してください。
2. `"2026-08-30"` を `"-"` で分割して、年・月・日をそれぞれ表示してください。
3. 商品名 `"りんご"`、単価 `128`、個数 `3` を変数にして、f 文字列で `りんご 3個: 384円` と表示してください。

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```python
# 1
s = "Hello, World"
print(s[7:12])      # または s[-5:]

# 2
parts = "2026-08-30".split("-")
print(parts[0], parts[1], parts[2])

# 3
item = "りんご"
price = 128
count = 3
print(f"{item} {count}個: {price * count}円")
```

</details>

---
## 4. リスト

**リスト** は複数の値を順番に並べて保存するデータ構造です。`[]` の中にカンマ区切りで値を書きます。
文字列と同じようにインデックスとスライスが使えます。

### 4.1 リストの作成とアクセス

In [ ]:
fruits = ["りんご", "バナナ", "みかん"]
numbers = [10, 20, 30, 40, 50]
mixed = [1, "二", 3.0, True]     # 異なる型を混ぜてもよい
empty = []                       # 空のリスト

print(fruits)
print(len(numbers))
print(fruits[0], fruits[-1])
print(numbers[1:4])
print(mixed, empty)

### 4.2 要素の追加・変更・削除

リストは文字列と違って **変更できる**（ミュータブル）データ構造です。

In [ ]:
fruits = ["りんご", "バナナ"]
fruits.append("みかん")             # 末尾に追加
print(fruits)
fruits.insert(1, "ぶどう")          # 位置を指定して挿入
print(fruits)
fruits.extend(["もも", "なし"])     # 複数まとめて追加
print(fruits)
fruits[0] = "青りんご"              # 要素の変更
print(fruits)

In [ ]:
fruits.remove("バナナ")     # 値を指定して削除
print(fruits)
last = fruits.pop()         # 末尾の要素を取り出して削除
print(last, fruits)
del fruits[0]               # インデックスを指定して削除
print(fruits)

### 4.3 便利な操作

In [ ]:
scores = [72, 95, 58, 88, 64]
print(sum(scores), max(scores), min(scores))
print(sum(scores) / len(scores))      # 平均
print(sorted(scores))                 # 並べ替えた新しいリストを返す（元は変わらない）
print(sorted(scores, reverse=True))   # 降順
print(scores)
scores.sort()                         # 元のリスト自体を並べ替える
print(scores)
print(95 in scores)                   # 含まれているか
print(scores.index(88))               # 位置

### 4.4 リストのコピーに注意

`b = a` と書いても新しいリストは作られず、**同じリストに別の名前が付くだけ** です。
別のリストが欲しいときは `copy()` を使います。

In [ ]:
a = [1, 2, 3]
b = a            # 同じリストを指す（コピーではない）
b.append(4)
print(a, b)      # a も変わってしまう

c = a.copy()     # 別のリストを作る
c.append(5)
print(a, c)      # a は変わらない

### 4.5 ネストしたリスト

リストの中にリストを入れると、表（2 次元）のようなデータを表せます。

In [ ]:
matrix = [[1, 2, 3],
          [4, 5, 6],
          [7, 8, 9]]
print(matrix[1])       # 2 行目
print(matrix[1][2])    # 2 行目の 3 列目
print(len(matrix), len(matrix[0]))   # 行数, 列数

### 練習問題 4

1. 1 から 5 までの整数のリストを作り、末尾に 6 を追加し、先頭の要素を削除して表示してください。
2. `[3, 1, 4, 1, 5, 9, 2, 6]` の最大値・最小値・合計・平均を表示してください。
3. 同じリストを降順に並べ替えて、上位 3 つだけを表示してください。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```python
# 1
nums = [1, 2, 3, 4, 5]
nums.append(6)
del nums[0]        # または nums.pop(0)
print(nums)

# 2
data = [3, 1, 4, 1, 5, 9, 2, 6]
print(max(data), min(data), sum(data), sum(data) / len(data))

# 3
print(sorted(data, reverse=True)[:3])
```

</details>

---
## 5. タプル・辞書・集合

リスト以外にも、値をまとめて扱うデータ構造があります。

| 種類 | 書き方 | 特徴 |
|---|---|---|
| リスト `list` | `[1, 2, 3]` | 順序あり、変更できる |
| タプル `tuple` | `(1, 2, 3)` | 順序あり、**変更できない** |
| 辞書 `dict` | `{"key": value}` | キーと値のペア |
| 集合 `set` | `{1, 2, 3}` | 重複なし、順序なし |

### 5.1 タプル

タプルは **変更できないリスト** です。座標や、関数から複数の値を返すときなどに使われます。

In [ ]:
point = (3, 4)
print(point[0], point[1])
print(type(point), len(point))
# point[0] = 10    # ← エラー（TypeError）。タプルは変更できない

# アンパック：要素を別々の変数に取り出す
x, y = point
print(x, y)

### 5.2 辞書

**辞書** は「キー」と「値」のペアを保存します。キーを指定して値を取り出せます。

In [ ]:
student = {"name": "田中", "age": 20, "major": "経済学"}
print(student["name"])
print(student.get("email"))               # 存在しないキーは None
print(student.get("email", "未登録"))      # デフォルト値を指定

In [ ]:
student["email"] = "tanaka@example.com"   # 追加
student["age"] = 21                        # 更新
print(student)
del student["major"]                       # 削除
print(student)
print("name" in student)                   # キーがあるか

In [ ]:
print(student.keys())      # キーの一覧
print(student.values())    # 値の一覧
print(student.items())     # (キー, 値) の一覧
print(len(student))

In [ ]:
# 辞書は「名前で引く表」として使うと便利
prices = {"りんご": 128, "バナナ": 98, "みかん": 60}
print("バナナ 3 本:", prices["バナナ"] * 3, "円")

### 5.3 集合

**集合** は重複のない値の集まりです。重複の除去や、和・積・差の計算に使います。

In [ ]:
nums = [1, 2, 2, 3, 3, 3]
unique = set(nums)
print(unique)            # 重複が消える
print(len(unique))

a = {1, 2, 3, 4}
b = {3, 4, 5, 6}
print(a | b)    # 和集合（どちらかにある）
print(a & b)    # 積集合（両方にある）
print(a - b)    # 差集合（a にあって b にない）

### 練習問題 5

1. 都道府県名をキー、県庁所在地を値とする辞書を 3 件分作り、`"愛知県"` の県庁所在地を表示してください。
2. その辞書にもう 1 件追加し、キーの一覧を表示してください。
3. `[1, 2, 2, 3, 4, 4, 5]` から重複を除いた要素の個数を表示してください。

In [ ]:
# 練習問題 5 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 5 の解答例を見る</strong></summary>

```python
# 1
capitals = {"愛知県": "名古屋市", "東京都": "新宿区", "大阪府": "大阪市"}
print(capitals["愛知県"])

# 2
capitals["北海道"] = "札幌市"
print(capitals.keys())

# 3
print(len(set([1, 2, 2, 3, 4, 4, 5])))
```

</details>

---
## 6. 条件分岐（if 文）

### 6.1 比較演算子と真偽値

条件は `True` または `False` になる式で表します。

| 演算子 | 意味 |
|:---:|---|
| `==` | 等しい（`=` 1 つは代入なので注意） |
| `!=` | 等しくない |
| `<` `<=` | より小さい、以下 |
| `>` `>=` | より大きい、以上 |

In [ ]:
x = 10
print(x > 5, x < 5)
print(x == 10, x != 10)
print(x >= 10, x <= 9)
print("abc" == "abc", "abc" < "abd")   # 文字列も比較できる（辞書順）

### 6.2 if / elif / else

```python
if 条件1:
    条件1 が True のときの処理
elif 条件2:
    条件1 が False で 条件2 が True のときの処理
else:
    どれにも当てはまらないときの処理
```

- 条件の後ろの **コロン `:`** を忘れずに
- 処理の部分は **半角スペース 4 つ** でインデント（字下げ）する。Python はインデントでブロックを判断します

In [ ]:
score = 78
if score >= 80:
    print("優")
elif score >= 60:
    print("良")
else:
    print("不可")

In [ ]:
temperature = 28
if temperature >= 25:
    print("暑い日です")
    print("水分補給を忘れずに")     # 同じインデントなので、同じブロック
print("今日の気温は", temperature, "度")   # if の外なので、常に実行される

### 6.3 論理演算子

複数の条件を組み合わせるには `and`（かつ）、`or`（または）、`not`（〜でない）を使います。

In [ ]:
age = 20
has_ticket = True

if age >= 18 and has_ticket:
    print("入場できます")

if age < 18 or not has_ticket:
    print("入場できません")
else:
    print("チェック完了")

In [ ]:
# 範囲の判定は数学のように書ける
n = 15
print(0 <= n <= 100)
print(10 < n < 12)

### 6.4 真と偽になる値

`if` の条件には、真偽値以外の値も書けます。`0`、空の文字列 `""`、空のリスト `[]`、`None` は **偽** として扱われ、
それ以外は **真** として扱われます。

In [ ]:
print(bool(0), bool(1))
print(bool(""), bool("a"))
print(bool([]), bool([1]))
print(bool(None))

items = []
if not items:
    print("リストは空です")

### 6.5 条件式（三項演算子）

`値1 if 条件 else 値2` と書くと、条件によって値を選ぶ式を 1 行で書けます。

In [ ]:
n = 7
parity = "偶数" if n % 2 == 0 else "奇数"
print(n, "は", parity)

### 練習問題 6

1. 変数 `year` がうるう年かどうかを判定して表示してください（4 で割り切れ、かつ 100 で割り切れない年、または 400 で割り切れる年がうるう年です）。
2. 身長 `height`（m）と体重 `weight`（kg）から BMI（体重 ÷ 身長²）を計算し、18.5 未満なら「低体重」、25 未満なら「普通体重」、それ以上なら「肥満」と表示してください。
3. 文字列 `text` が空なら「入力なし」、そうでなければ文字数を表示してください。

In [ ]:
# 練習問題 6 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 6 の解答例を見る</strong></summary>

```python
# 1
year = 2024
if (year % 4 == 0 and year % 100 != 0) or year % 400 == 0:
    print(year, "年はうるう年です")
else:
    print(year, "年はうるう年ではありません")

# 2
height = 1.70
weight = 65
bmi = weight / height ** 2
if bmi < 18.5:
    print("低体重")
elif bmi < 25:
    print("普通体重")
else:
    print("肥満")

# 3
text = "Python"
if not text:
    print("入力なし")
else:
    print(len(text), "文字")
```

</details>

---
## 7. 繰り返し（for 文・while 文）

### 7.1 for 文と range

`for 変数 in 値の並び:` と書くと、値を 1 つずつ取り出して同じ処理を繰り返します。
`range(n)` は 0 から n-1 までの整数の並びを作ります。

In [ ]:
for i in range(5):
    print(i)

In [ ]:
print(list(range(5)))          # 0〜4
print(list(range(1, 6)))       # 1〜5（開始, 終了）
print(list(range(0, 10, 2)))   # 0〜8 を 2 刻み（開始, 終了, 刻み）
print(list(range(10, 0, -3)))  # 減らすこともできる

In [ ]:
total = 0
for i in range(1, 11):
    total += i
print("1 から 10 の合計:", total)

### 7.2 リストや文字列の繰り返し

リスト・文字列・辞書など、「値の並び」ならなんでも `for` で回せます。

In [ ]:
fruits = ["りんご", "バナナ", "みかん"]
for fruit in fruits:
    print(fruit, len(fruit))

for ch in "abc":
    print(ch.upper())

### 7.3 enumerate と zip

- `enumerate()`：番号と要素を同時に取り出す
- `zip()`：複数のリストを同時に回す

In [ ]:
for i, fruit in enumerate(fruits):
    print(i, fruit)

for i, fruit in enumerate(fruits, start=1):
    print(f"{i}番目: {fruit}")

In [ ]:
names = ["田中", "鈴木", "佐藤"]
scores = [85, 92, 78]
for name, score in zip(names, scores):
    print(f"{name}: {score}点")

### 7.4 辞書の繰り返し

In [ ]:
prices = {"りんご": 128, "バナナ": 98, "みかん": 60}
for key in prices:               # キーだけ
    print(key)

for key, value in prices.items():   # キーと値
    print(f"{key} は {value} 円")

### 7.5 while 文

`while 条件:` は、条件が `True` の間ずっと繰り返します。繰り返す回数が決まっていないときに使います。
条件がいつまでも `False` にならないと **無限ループ** になるので注意してください
（止まらなくなったら、メニューの Kernel → Interrupt Kernel で中断できます）。

In [ ]:
count = 0
while count < 3:
    print("count =", count)
    count += 1
print("終了")

In [ ]:
# 例：年利 5% で預金が 2 倍になるまでの年数
balance = 100
years = 0
while balance < 200:
    balance *= 1.05
    years += 1
print(f"{years} 年後に {balance:.1f} になりました")

### 7.6 break と continue

- `break`：ループを途中で終了する
- `continue`：今回の残りの処理を飛ばして次の繰り返しへ進む

In [ ]:
for i in range(10):
    if i == 7:
        break          # ループを抜ける
    if i % 2 == 0:
        continue       # 偶数のときは以降をスキップ
    print(i)

### 7.7 ネストしたループ

ループの中にループを書くこともできます。

In [ ]:
for i in range(1, 4):
    for j in range(1, 4):
        print(f"{i * j:3d}", end="")
    print()    # 1 行分が終わったら改行

### 7.8 リスト内包表記

「リストの各要素に処理をして新しいリストを作る」操作は、**リスト内包表記** で 1 行に書けます。

In [ ]:
# for 文で書く
squares = []
for i in range(1, 6):
    squares.append(i ** 2)
print(squares)

# リスト内包表記で書く（同じ結果）
squares = [i ** 2 for i in range(1, 6)]
print(squares)

# 条件付き
evens = [i for i in range(10) if i % 2 == 0]
print(evens)

### 練習問題 7

1. `for` 文を使って 1 から 100 までの偶数の合計を求めてください。
2. 1 から 15 までの数について、3 の倍数なら「Fizz」、5 の倍数なら「Buzz」、両方の倍数なら「FizzBuzz」、それ以外はその数を表示してください（FizzBuzz 問題）。
3. 九九の 5 の段を `5 x 1 = 5` の形式で表示してください。
4. リスト内包表記で、1 から 30 までの 3 の倍数のリストを作ってください。

In [ ]:
# 練習問題 7 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 7 の解答例を見る</strong></summary>

```python
# 1
total = 0
for i in range(1, 101):
    if i % 2 == 0:
        total += i
print(total)

# 2
for i in range(1, 16):
    if i % 15 == 0:
        print("FizzBuzz")
    elif i % 3 == 0:
        print("Fizz")
    elif i % 5 == 0:
        print("Buzz")
    else:
        print(i)

# 3
for i in range(1, 10):
    print(f"5 x {i} = {5 * i}")

# 4
multiples = [i for i in range(1, 31) if i % 3 == 0]
print(multiples)
```

</details>

---
## 8. 関数

**関数** は、ひとまとまりの処理に名前を付けて再利用できるようにしたものです。
`print()` や `len()` も関数ですが、自分で作ることもできます。

```python
def 関数名(引数1, 引数2, ...):
    処理
    return 戻り値
```

### 8.1 関数の定義と呼び出し

In [ ]:
def greet():
    print("こんにちは！")

greet()      # 呼び出し
greet()      # 何度でも使える

In [ ]:
def greet(name):               # name は引数（関数に渡す値）
    print(f"こんにちは、{name}さん！")

greet("山田")
greet("鈴木")

### 8.2 戻り値

`return` で計算結果を呼び出し元に返せます。返した値は変数に入れたり、別の計算に使ったりできます。

In [ ]:
def add(a, b):
    return a + b

result = add(3, 4)
print(result)
print(add(10, 20) * 2)

In [ ]:
def circle_area(radius):
    """半径から円の面積を求める。（この文字列は docstring：関数の説明）"""
    pi = 3.14159
    return pi * radius ** 2

print(circle_area(2))
print(circle_area.__doc__)

### 8.3 デフォルト引数とキーワード引数

- 引数に `= 値` を付けると、省略したときの **デフォルト値** になります
- 呼び出すときに `引数名=値` と書くと、順番に関係なく指定できます（**キーワード引数**）

In [ ]:
def introduce(name, age=20, city="名古屋"):
    print(f"{name}（{age}歳、{city}在住）")

introduce("田中")
introduce("鈴木", 25)
introduce("佐藤", city="東京")
introduce(age=30, name="高橋")

### 8.4 複数の値を返す

`return a, b` のようにカンマで区切ると、タプルとしてまとめて返せます。

In [ ]:
def min_max(values):
    return min(values), max(values)

lo, hi = min_max([3, 8, 1, 9, 4])
print(lo, hi)

### 8.5 変数のスコープ（有効範囲）

関数の中で作った変数は **ローカル変数** と呼ばれ、関数の外からは見えません。
関数の外で作った変数は **グローバル変数** で、関数の中から読むことはできます。

In [ ]:
message = "グローバル変数"

def show():
    message = "ローカル変数"    # 関数の中だけで有効な別の変数
    print("関数の中:", message)

show()
print("関数の外:", message)      # 外の変数は変わっていない

### 8.6 lambda 式（無名関数）

`lambda 引数: 式` で、名前のない小さな関数を作れます。並べ替えの基準を指定するときなどに便利です。

In [ ]:
square = lambda x: x ** 2
print(square(5))

# sorted の key に使う例：年齢順に並べ替える
people = [("田中", 25), ("鈴木", 19), ("佐藤", 32)]
print(sorted(people, key=lambda p: p[1]))

### 8.7 よく使う組み込み関数

Python に最初から用意されている関数を **組み込み関数** といいます。

| 関数 | 説明 |
|---|---|
| `print()`, `len()`, `type()` | 表示、長さ、型 |
| `int()`, `float()`, `str()`, `bool()`, `list()` | 型変換 |
| `range()`, `enumerate()`, `zip()` | 繰り返し用 |
| `sum()`, `max()`, `min()`, `sorted()`, `reversed()` | 集計・並べ替え |
| `abs()`, `round()`, `pow()` | 絶対値、四捨五入、べき乗 |
| `isinstance()` | 型の判定 |

In [ ]:
print(abs(-5), round(2.567, 1), pow(2, 5))
print(max("apple", "banana"), min([3, 1, 2]))
print(list(reversed([1, 2, 3])))
print(isinstance(3, int), isinstance("3", int))

### 練習問題 8

1. 摂氏温度を華氏温度に変換する関数 `c_to_f(celsius)` を作り、`c_to_f(25)` の結果を表示してください（華氏 = 摂氏 × 9 / 5 + 32）。
2. リストの平均を返す関数 `average(values)` を作ってください。リストが空のときは `0` を返します。
3. 税込価格を返す関数 `with_tax(price, rate=0.1)` を作り、`with_tax(1000)` と `with_tax(1000, rate=0.08)` を表示してください。

In [ ]:
# 練習問題 8 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 8 の解答例を見る</strong></summary>

```python
# 1
def c_to_f(celsius):
    return celsius * 9 / 5 + 32

print(c_to_f(25))

# 2
def average(values):
    if not values:
        return 0
    return sum(values) / len(values)

print(average([80, 90, 70]), average([]))

# 3
def with_tax(price, rate=0.1):
    return int(price * (1 + rate))

print(with_tax(1000), with_tax(1000, rate=0.08))
```

</details>

---
## 9. エラーと例外処理

### 9.1 エラーメッセージの読み方

プログラムにミスがあると、赤い枠でエラーメッセージ（**トレースバック**）が表示されます。
慌てずに、次の 2 点を確認しましょう。

1. **最後の行** にエラーの種類と説明が書かれている（例: `IndexError: list index out of range`）
2. `---->` の矢印が付いた行が、エラーが起きた場所

| エラーの種類 | 主な原因 |
|---|---|
| `SyntaxError` | 文法の間違い（かっこや `:` の閉じ忘れなど） |
| `IndentationError` | インデントがそろっていない |
| `NameError` | 定義していない変数・関数を使った（実行順やスペルミスに注意） |
| `TypeError` | 型が合わない操作（`1 + "2"` など） |
| `ValueError` | 型は合っているが値が不適切（`int("abc")` など） |
| `IndexError` | リストの範囲外のインデックスを指定した |
| `KeyError` | 辞書に存在しないキーを指定した |
| `ZeroDivisionError` | 0 で割った |
| `ModuleNotFoundError` | ライブラリがインストールされていない（0.2 節を参照） |

次のセルのコメントを外して実行すると、実際にエラーを見ることができます（確認したら元に戻してください）。

In [ ]:
# わざとエラーを起こしてみる（1 行ずつコメントを外して試してみましょう）
# numbers = [1, 2, 3]
# print(numbers[5])          # IndexError
# print(undefined_variable)  # NameError
# print(int("abc"))          # ValueError
print("エラーが出なければ、このメッセージが表示されます")

### 9.2 try / except

エラーが起きそうな処理を `try:` の中に書き、`except エラーの種類:` で対処を書くと、
プログラムを止めずに続行できます。これを **例外処理** といいます。

In [ ]:
try:
    result = 10 / 0
except ZeroDivisionError:
    print("0 で割ることはできません")

In [ ]:
def to_int(text):
    try:
        return int(text)
    except ValueError:
        print(f"'{text}' は整数に変換できません")
        return None

print(to_int("42"))
print(to_int("abc"))

In [ ]:
# as でエラーの情報を受け取る / else はエラーがなかったとき / finally は常に実行
data = {"a": 1}
for key in ["a", "b"]:
    try:
        value = data[key]
    except KeyError as e:
        print("キーがありません:", e)
    else:
        print("値:", value)
    finally:
        print("--- 処理終了 ---")

### 9.3 raise：自分でエラーを発生させる

不正な値を受け取ったときなどに、`raise` で意図的にエラーを起こせます。

In [ ]:
def set_age(age):
    if age < 0:
        raise ValueError("年齢は 0 以上で指定してください")
    return age

try:
    set_age(-1)
except ValueError as e:
    print("エラー:", e)

### 練習問題 9

1. 文字列のリスト `["10", "abc", "30", "4.5"]` の各要素を整数に変換して合計してください。変換できない要素は表示してスキップします。
2. 辞書 `d` とキー `key` を受け取り、値があればその値を、なければ `"不明"` を返す関数 `lookup(d, key)` を `try / except` を使って作ってください。

In [ ]:
# 練習問題 9 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 9 の解答例を見る</strong></summary>

```python
# 1
texts = ["10", "abc", "30", "4.5"]
total = 0
for t in texts:
    try:
        total += int(t)
    except ValueError:
        print(f"{t!r} はスキップしました")
print("合計:", total)

# 2
def lookup(d, key):
    try:
        return d[key]
    except KeyError:
        return "不明"

capitals = {"愛知県": "名古屋市"}
print(lookup(capitals, "愛知県"), lookup(capitals, "岐阜県"))
```

</details>

---
## 10. モジュールと標準ライブラリ

**モジュール** は、関数や変数をまとめたファイルです。Python に付属するモジュール群を **標準ライブラリ** といい、
`import` するだけで使えます（0.2 節の (1) にあたります）。

### 10.1 import の書き方

In [ ]:
import math                     # モジュール全体を読み込む
print(math.sqrt(16), math.pi)
print(math.floor(3.7), math.ceil(3.2))   # 切り捨て、切り上げ

from math import sqrt, pi       # 必要な名前だけ取り込む
print(sqrt(2), pi)

import statistics as stats      # 別名を付けて読み込む
print(stats.mean([1, 2, 3, 4]), stats.median([1, 3, 2, 10]))

### 10.2 random：乱数

`random.seed()` で種を固定すると、毎回同じ乱数の並びになり、結果を再現できます。

In [ ]:
import random

random.seed(0)                              # 乱数の種を固定（再現性のため）
print(random.randint(1, 6))                 # 1〜6 の整数（サイコロ）
print(random.random())                      # 0 以上 1 未満の小数
print(random.choice(["グー", "チョキ", "パー"]))   # ランダムに 1 つ選ぶ
print(sorted(random.sample(range(1, 44), 6)))    # 重複なしで 6 個選ぶ（ロト 6）

### 10.3 datetime：日付と時刻

In [ ]:
from datetime import date, datetime, timedelta

today = date.today()
print(today)
print(today.year, today.month, today.day)
print(today + timedelta(days=100))          # 100 日後

dt = datetime(2026, 4, 1, 9, 30)
print(dt.strftime("%Y年%m月%d日 %H:%M"))     # 書式を指定して文字列に
print(datetime.now().strftime("%H:%M:%S"))  # 現在時刻

### 10.4 collections：数え上げ

In [ ]:
from collections import Counter

words = "a b a c b a".split()
counter = Counter(words)
print(counter)
print(counter.most_common(1))    # 最も多いもの

### 10.5 よく使う標準ライブラリ

| モジュール | 用途 |
|---|---|
| `math` | 数学関数（平方根、三角関数、対数など） |
| `random` | 乱数 |
| `statistics` | 平均・中央値・標準偏差など |
| `datetime` | 日付と時刻 |
| `collections` | Counter、defaultdict などの便利なデータ構造 |
| `csv` | CSV ファイルの読み書き（第 11 章） |
| `json` | JSON データの読み書き |
| `os`, `pathlib` | ファイルやフォルダの操作 |
| `re` | 正規表現（パターンによる文字列検索） |
| `time` | 処理時間の計測、待機 |

データ分析でよく使う NumPy や pandas は標準ライブラリではないので、JupyterLite では 0.2 節の方法で読み込みます。

### 練習問題 10

1. `math` モジュールを使って 2 の平方根を小数第 3 位まで表示してください。
2. サイコロを 10 回振った結果をリストに入れ、その平均を表示してください（`random.seed(1)` を使って再現できるようにしてください）。
3. 今日から 30 日後の日付を表示してください。

In [ ]:
# 練習問題 10 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 10 の解答例を見る</strong></summary>

```python
# 1
import math
print(f"{math.sqrt(2):.3f}")

# 2
import random
random.seed(1)
rolls = [random.randint(1, 6) for _ in range(10)]
print(rolls, sum(rolls) / len(rolls))

# 3
from datetime import date, timedelta
print(date.today() + timedelta(days=30))
```

</details>

---
## 11. ファイルの読み書き

`open()` でファイルを開き、`with` 文と組み合わせると、処理が終わったときに自動でファイルが閉じられます。

```python
with open("ファイル名", モード, encoding="utf-8") as f:
    処理
```

| モード | 意味 |
|:---:|---|
| `"r"` | 読み込み（省略時） |
| `"w"` | 書き込み（既存の内容は消える） |
| `"a"` | 追記 |

日本語を含むファイルは `encoding="utf-8"` を指定しておくと文字化けを防げます。

JupyterLite で作成したファイルはブラウザのローカルストレージに保存され、左側のファイルブラウザに表示されます
（表示されないときは、ファイルブラウザ上部の更新ボタンを押してください）。

### 11.1 テキストファイルの書き込み

In [ ]:
with open("sample.txt", "w", encoding="utf-8") as f:
    f.write("1行目: Python の練習\n")
    f.write("2行目: ファイルに書き込み\n")
print("sample.txt に書き込みました")

### 11.2 テキストファイルの読み込み

In [ ]:
with open("sample.txt", "r", encoding="utf-8") as f:
    content = f.read()        # 全体を 1 つの文字列として読む
print(content)

In [ ]:
with open("sample.txt", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):    # 1 行ずつ読む
        print(line_no, line.rstrip())              # 末尾の改行を取り除いて表示

In [ ]:
# 追記モード
with open("sample.txt", "a", encoding="utf-8") as f:
    f.write("3行目: 追記しました\n")

with open("sample.txt", encoding="utf-8") as f:
    print(f.read())

### 11.3 CSV ファイル

表形式のデータは CSV（カンマ区切り）ファイルで扱うことが多く、標準ライブラリの `csv` モジュールで読み書きできます。

In [ ]:
import csv

rows = [
    ["名前", "国語", "数学"],
    ["田中", 80, 75],
    ["鈴木", 92, 88],
    ["佐藤", 67, 95],
]
with open("scores.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(rows)
print("scores.csv に書き込みました")

In [ ]:
with open("scores.csv", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)          # 各行は文字列のリストになる

In [ ]:
# DictReader を使うと、1 行目の見出しをキーにした辞書として読める
with open("scores.csv", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        total = int(row["国語"]) + int(row["数学"])   # 読み込んだ値は文字列なので整数に変換
        print(row["名前"], total)

### 11.4 ファイルの存在確認と一覧

In [ ]:
import os

print(os.path.exists("sample.txt"))
print(os.path.exists("not_found.txt"))
print(sorted(os.listdir(".")))     # 現在のフォルダにあるファイル

### 練習問題 11

1. `"memo.txt"` に 3 行のメモを書き込み、読み込んで行数を表示してください。
2. `"scores.csv"` を読み込み、各人の合計点と、全員の合計点の平均を表示してください。

In [ ]:
# 練習問題 11 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 11 の解答例を見る</strong></summary>

```python
# 1
with open("memo.txt", "w", encoding="utf-8") as f:
    f.write("牛乳を買う\n")
    f.write("レポートを提出する\n")
    f.write("Python を復習する\n")

with open("memo.txt", encoding="utf-8") as f:
    lines = f.readlines()
print(len(lines), "行")

# 2
import csv
totals = []
with open("scores.csv", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        total = int(row["国語"]) + int(row["数学"])
        totals.append(total)
        print(row["名前"], total)
print("平均:", sum(totals) / len(totals))
```

</details>

---
## 12. クラス入門

これまで使ってきた文字列やリストは、実は **クラス** から作られた **オブジェクト** です
（`"abc".upper()` のように、値に付属するメソッドを呼び出せたのはそのためです）。
自分でクラスを定義すると、データ（**属性**）と処理（**メソッド**）をひとまとめにした独自の型を作れます。

```python
class クラス名:
    def __init__(self, 引数...):     # オブジェクトを作るときに呼ばれる（初期化）
        self.属性 = 値

    def メソッド名(self, 引数...):
        処理
```

- `self` は「このオブジェクト自身」を表し、メソッドの最初の引数に必ず書きます
- クラス名は `Student` のように単語の先頭を大文字にする（**キャメルケース**）のが慣習です

In [ ]:
class Student:
    def __init__(self, name, scores):
        self.name = name          # 属性
        self.scores = scores

    def average(self):            # メソッド
        return sum(self.scores) / len(self.scores)

    def introduce(self):
        print(f"{self.name}さんの平均点は {self.average():.1f} 点です")


s1 = Student("田中", [80, 75, 90])    # オブジェクトを作る（__init__ が呼ばれる）
s2 = Student("鈴木", [92, 88, 79])
s1.introduce()
s2.introduce()
print(type(s1))

In [ ]:
# 属性は読み書きできる
print(s1.name)
s1.scores.append(100)
s1.introduce()

`__str__` メソッドを定義すると、`print()` したときの表示を決められます。

In [ ]:
class Item:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def __str__(self):
        return f"{self.name}（{self.price}円）"


item = Item("ノート", 150)
print(item)

### 練習問題 12

1. 幅 `width` と高さ `height` を持つ `Rectangle` クラスを作り、面積を返す `area()` メソッドと周囲の長さを返す `perimeter()` メソッドを定義してください。`Rectangle(3, 4)` で試してみましょう。

In [ ]:
# 練習問題 12 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 12 の解答例を見る</strong></summary>

```python
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)


r = Rectangle(3, 4)
print(r.area(), r.perimeter())
```

</details>

---
## 13. まとめ

このチュートリアルで学んだことをまとめます。

| トピック | 主な文法・関数 |
|---|---|
| JupyterLite の準備 | `piplite.install()`, `%pip install`, `import japanize_matplotlib_jlite` |
| 基本 | `print()`, `#` コメント, 算術演算子 `+ - * / // % **` |
| 変数とデータ型 | `int`, `float`, `str`, `bool`, `type()`, `int()`, `float()`, `str()` |
| 文字列 | インデックス・スライス, `strip()`, `split()`, `replace()`, `join()`, f 文字列 |
| リスト | `append()`, `insert()`, `remove()`, `pop()`, `sort()`, `sorted()`, `len()`, `sum()` |
| タプル・辞書・集合 | `(a, b)`, `{"key": value}`, `get()`, `items()`, `set()`, `| & -` |
| 条件分岐 | `if / elif / else`, 比較演算子, `and / or / not` |
| 繰り返し | `for`, `range()`, `enumerate()`, `zip()`, `while`, `break`, `continue`, 内包表記 |
| 関数 | `def`, `return`, デフォルト引数, キーワード引数, `lambda` |
| 例外処理 | `try / except / else / finally`, `raise` |
| モジュール | `import`, `from ... import`, `math`, `random`, `datetime`, `collections` |
| ファイル | `with open()`, `read()`, `write()`, `csv` モジュール |
| クラス | `class`, `__init__`, `self`, メソッド, `__str__` |

## 次のステップ

Python の基本が身についたら、データ分析用のライブラリに進みましょう。

1. `python/numpy/numpy_beginner_tutorial.ipynb` — 配列計算の基礎
2. `python/pandas/pandas_beginner_tutorial.ipynb` — 表データの操作
3. `python/matplotlib/matplotlib_beginner_tutorial.ipynb` — グラフの作成
4. `exercises/python_beginner_exercises_34.ipynb` — 総合練習問題

---
## 総合演習：成績管理プログラム

これまで学んだ内容を組み合わせて、次の課題に挑戦してください。

次の辞書は、4 人の学生の 3 科目の点数です。

```python
scores = {
    "田中": {"国語": 78, "数学": 92, "英語": 85},
    "鈴木": {"国語": 88, "数学": 64, "英語": 71},
    "佐藤": {"国語": 95, "数学": 89, "英語": 93},
    "高橋": {"国語": 55, "数学": 71, "英語": 60},
}
```

1. 平均点から評価を返す関数 `grade(avg)` を作ってください（85 以上「優」、70 以上「良」、60 以上「可」、それ未満「不可」）。
2. 学生ごとに合計点・平均点・評価を計算し、`田中: 合計 255 点, 平均 85.0 点, 評価 優` の形式で表示してください。
3. 科目ごとの平均点を計算し、最も平均点が高い科目を表示してください。
4. 学生ごとの結果（名前・合計・平均・評価）を `results.csv` に保存してください。
5. 学生ごとの平均点を棒グラフにしてください（タイトルや軸ラベルは日本語で。0.3 節の日本語フォント設定を使います）。

In [ ]:
# 総合演習の解答欄：ここにコードを書いてください
scores = {
    "田中": {"国語": 78, "数学": 92, "英語": 85},
    "鈴木": {"国語": 88, "数学": 64, "英語": 71},
    "佐藤": {"国語": 95, "数学": 89, "英語": 93},
    "高橋": {"国語": 55, "数学": 71, "英語": 60},
}

### 総合演習の解答例

自分で書いてから、次のセルを実行して結果を比べてみてください。

In [ ]:
import csv
import matplotlib.pyplot as plt
import japanize_matplotlib_jlite  # 日本語フォント

scores = {
    "田中": {"国語": 78, "数学": 92, "英語": 85},
    "鈴木": {"国語": 88, "数学": 64, "英語": 71},
    "佐藤": {"国語": 95, "数学": 89, "英語": 93},
    "高橋": {"国語": 55, "数学": 71, "英語": 60},
}


# 1. 評価を返す関数
def grade(avg):
    if avg >= 85:
        return "優"
    elif avg >= 70:
        return "良"
    elif avg >= 60:
        return "可"
    else:
        return "不可"


# 2. 学生ごとの集計
results = []
for name, subjects in scores.items():
    total = sum(subjects.values())
    avg = total / len(subjects)
    results.append({"名前": name, "合計": total, "平均": round(avg, 1), "評価": grade(avg)})
    print(f"{name}: 合計 {total} 点, 平均 {avg:.1f} 点, 評価 {grade(avg)}")

# 3. 科目ごとの平均
subject_avg = {}
for subject in ["国語", "数学", "英語"]:
    values = [s[subject] for s in scores.values()]
    subject_avg[subject] = sum(values) / len(values)
best_subject = max(subject_avg, key=subject_avg.get)
print("科目別平均:", subject_avg)
print("最も平均点が高い科目:", best_subject)

# 4. CSV に保存
with open("results.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["名前", "合計", "平均", "評価"])
    writer.writeheader()
    writer.writerows(results)
print("results.csv に保存しました")

# 5. 棒グラフ
names = [r["名前"] for r in results]
avgs = [r["平均"] for r in results]
overall = sum(avgs) / len(avgs)

plt.figure(figsize=(7, 4))
plt.bar(names, avgs, color="steelblue")
plt.axhline(overall, color="red", linestyle="--", label=f"全体平均 {overall:.1f} 点")
plt.title("学生別の平均点")
plt.xlabel("学生")
plt.ylabel("平均点")
plt.ylim(0, 100)
plt.legend()
plt.show()

お疲れさまでした！ ここまでできれば、Python の基本文法はひととおり身についています。
「次のステップ」のノートブックに進んで、NumPy や pandas を使ったデータ分析に挑戦してみましょう。